In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
# Load and clean
df = pd.read_csv('uber.csv')
df = df.drop(columns=['Unnamed: 0', 'key'], errors='ignore')
df = df.dropna()
df.head()

In [ ]:
# Feature engineering: hour, weekday, distance
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.dayofweek

dx = df['pickup_longitude'] - df['dropoff_longitude']
dy = df['pickup_latitude'] - df['dropoff_latitude']
df['distance_km'] = np.sqrt(dx**2 + dy**2) * 111   # 1 degree ~ 111 km

df = df.drop(columns=['pickup_datetime']).dropna()

In [ ]:
# Identify outliers (boxplot) and remove them with the IQR rule
df.boxplot(column=['fare_amount', 'distance_km'])
plt.show()

for col in ['fare_amount', 'distance_km']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

df = df[(df['fare_amount'] > 0) & (df['passenger_count'] >= 1) & (df['passenger_count'] <= 6)]
print('Rows after cleaning:', len(df))

In [ ]:
# Correlation
df.corr()['fare_amount']

In [ ]:
# Train / test split
X = df.drop(columns=['fare_amount'])
y = df['fare_amount']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

print('Linear Regression')
print('R2   :', r2_score(y_test, pred_lr))
print('RMSE :', np.sqrt(mean_squared_error(y_test, pred_lr)))

In [ ]:
# Random Forest Regression
rf = RandomForestRegressor(n_estimators=50, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print('Random Forest')
print('R2   :', r2_score(y_test, pred_rf))
print('RMSE :', np.sqrt(mean_squared_error(y_test, pred_rf)))